# Tkinter – Clase 2 ampliada: widgets, variables, eventos y `ttk`



**Situación inicial:** una farmacia necesita una pequeña interfaz para registrar pedidos, elegir el turno de atención y anotar observaciones. Hoy construiremos sus piezas y luego un bloc de notas capaz de abrir y guardar archivos.

**Requisitos:** Python 3 con Tkinter y Jupyter en un equipo con escritorio gráfico. Ejecuta **una celda de ventana por vez**: ciérrala antes de pasar a la siguiente, porque `mainloop()` mantiene la ejecución hasta cerrarla. Para conservar una aplicación, copia su celda a un archivo `.py` y ejecútalo con Python.

**Resultados esperados:** leer y validar entradas; conectar widgets con variables; distinguir `command` de `bind`; usar `ttk`, cuadros de diálogo y archivos; construir una aplicación integradora.

## 1. Lo que ya sabemos: del programa a la ventana 

Una interfaz gráfica responde a acciones del usuario. `Tk()` crea la ventana principal; los widgets muestran o reciben información; `pack()` y `grid()` los sitúan; `mainloop()` espera eventos. La función indicada en `command` se ejecuta cuando ocurre la acción: escribe `command=saludar`, sin paréntesis. Si escribes `saludar()`, se ejecutará inmediatamente al construir el botón.

**Predicción:** ¿qué cambiará al hacer clic dos veces?

In [ ]:
import tkinter as tk

root = tk.Tk()
root.title("Repaso: mostrador")
root.geometry("380x160")
contador = 0

def saludar():
    global contador
    contador += 1
    etiqueta.config(text=f"Pedidos atendidos: {contador}")

etiqueta = tk.Label(root, text="Pedidos atendidos: 0")
etiqueta.pack(pady=16)
tk.Button(root, text="Atender pedido", command=saludar).pack()
root.mainloop()

**Observa:** el valor de `contador` cambia al pulsar el botón; `.config()` actualiza la etiqueta. En proyectos mayores, una clase o una variable Tkinter permite organizar mejor el estado. Revisa que no haya errores en la consola antes de avanzar.

## 2. Variables de Tkinter y entradas 

### Conceptos

| Elemento | Para qué sirve | Lectura / escritura |
|---|---|---|
| `StringVar` | Texto vinculado a un widget | `.get()` / `.set(texto)` |
| `IntVar` | Entero, por ejemplo una casilla 0/1 | `.get()` / `.set(numero)` |
| `Entry` | Una línea | `.get()` o `textvariable=...` |
| `Text` | Varias líneas | `.get("1.0", "end-1c")` |

Las variables Tkinter se crean **después** de `Tk()`. `Text` no admite `textvariable`: se lee por índices; `1.0` significa primera línea, carácter cero y `end-1c` evita el salto de línea final automático. `strip()` elimina espacios sobrantes al principio y al final; un texto compuesto solo por espacios queda vacío.

**Ejemplo guiado:** registra el nombre de un cliente y muestra un resumen de su pedido.

In [ ]:
import tkinter as tk
from tkinter import messagebox

root = tk.Tk()
root.title("Pedido: entradas")
root.geometry("460x340")
nombre_var = tk.StringVar()

tk.Label(root, text="Nombre del cliente:").pack(anchor="w", padx=12, pady=(12, 2))
tk.Entry(root, textvariable=nombre_var, width=40).pack(anchor="w", padx=12)
tk.Label(root, text="Detalle del pedido:").pack(anchor="w", padx=12, pady=(12, 2))
detalle = tk.Text(root, height=6, width=48)
detalle.pack(padx=12)

def mostrar():
    nombre = nombre_var.get().strip()
    pedido = detalle.get("1.0", "end-1c").strip()
    if not nombre or not pedido:
        messagebox.showwarning("Datos incompletos", "Escribe nombre y detalle.")
        return
    messagebox.showinfo("Pedido", f"Cliente: {nombre}\nDetalle: {pedido}")

tk.Button(root, text="Ver resumen", command=mostrar).pack(pady=12)
root.mainloop()

**Lectura del código:** `textvariable` sincroniza `Entry` con `nombre_var`; `mostrar()` lee ambos campos cuando se pulsa el botón; `return` detiene la función si faltan datos. Prueba nombres con espacios y un pedido vacío.

**Práctica breve (8 min):** agrega un campo `Entry` para teléfono. Exige que no esté vacío; muestra el dato en el resumen. Para aceptar únicamente dígitos puedes comprobar `telefono.isdigit()`; considera que algunos teléfonos admiten prefijos y signos, así que define primero la regla que necesita tu actividad.

## 3. Selección y widgets `ttk` 

Un grupo de `Radiobutton` comparte **una** variable: solo una opción queda elegida. `Checkbutton` representa una decisión independiente, por ejemplo “entrega a domicilio”. `ttk` aporta widgets con apariencia integrada al sistema; `Combobox(state="readonly")` limita la selección a los valores ofrecidos.

| Widget | Ejemplo de pregunta | Variable |
|---|---|---|
| `Radiobutton` | ¿En qué turno? | `StringVar` compartida |
| `Checkbutton` | ¿Necesita entrega? | `BooleanVar` |
| `ttk.Combobox` | ¿Qué categoría? | `StringVar` |

**Ejemplo integrado:**

In [1]:
import tkinter as tk
from tkinter import ttk, messagebox

root = tk.Tk()
root.title("Opciones del pedido")
root.geometry("400x310")
turno = tk.StringVar(value="Mañana")
entrega = tk.BooleanVar(value=False)
categoria = tk.StringVar(value="Seleccionar...")

ttk.Label(root, text="Turno:").pack(anchor="w", padx=12, pady=(12, 0))
for opcion in ("Mañana", "Tarde", "Noche"):
    ttk.Radiobutton(root, text=opcion, variable=turno, value=opcion).pack(anchor="w", padx=12)
ttk.Checkbutton(root, text="Entrega a domicilio", variable=entrega).pack(anchor="w", padx=12, pady=8)
ttk.Label(root, text="Categoría:").pack(anchor="w", padx=12)
combo = ttk.Combobox(root, textvariable=categoria, values=("Higiene", "Cuidado personal", "Otros"), state="readonly")
combo.pack(anchor="w", padx=12, pady=4)

def confirmar():
    if categoria.get() == "Seleccionar...":
        messagebox.showwarning("Falta una categoría", "Selecciona una categoría.")
        return
    messagebox.showinfo("Resumen", f"Turno: {turno.get()}\nEntrega: {'Sí' if entrega.get() else 'No'}\nCategoría: {categoria.get()}")

ttk.Button(root, text="Confirmar", command=confirmar).pack(pady=12)
root.mainloop()

**Observa:** `value` distingue las opciones de radio; `BooleanVar.get()` devuelve `True` o `False`; el texto inicial del combo exige seleccionar una categoría real. **Modifica:** agrega la categoría “Botiquín” y comprueba la selección.

## 4. Eventos y validación 

`command=funcion` es cómodo para botones. `bind("<Return>", funcion)` conecta una tecla y entrega un objeto `event` a la función; `event.widget` identifica el widget y en un clic también hay `event.x` e `event.y`. Para usar el mismo callback con botón y Enter, acepta `event=None`. La validación debe comunicar qué corregir sin cerrar la ventana.

**Ejemplo:** búsqueda simple en una lista local.

In [2]:
import tkinter as tk
from tkinter import ttk

productos = ["Alcohol", "Algodón", "Jabón", "Termómetro"]
root = tk.Tk()
root.title("Buscar productos")
root.geometry("410x190")
busqueda = tk.StringVar()

ttk.Label(root, text="Escribe un producto y pulsa Enter o Buscar:").pack(pady=8)
campo = ttk.Entry(root, textvariable=busqueda, width=32)
campo.pack()
resultado = ttk.Label(root, text="Esperando consulta")
resultado.pack(pady=12)

def buscar(event=None):
    texto = busqueda.get().strip().casefold()
    if not texto:
        resultado.config(text="Escribe un nombre antes de buscar.")
        return
    hallados = [p for p in productos if texto in p.casefold()]
    resultado.config(text=", ".join(hallados) if hallados else "Sin coincidencias")

ttk.Button(root, text="Buscar", command=buscar).pack()
campo.bind("<Return>", buscar)
campo.focus_set()
root.mainloop()

**Razonamiento:** `casefold()` permite comparar sin depender de mayúsculas; `in` encuentra fragmentos; la lista se filtra al pulsar el botón o Enter. **Prueba:** “al”, “jabón”, espacios y una palabra inexistente. **Extensión:** muestra cuántas coincidencias hay.

Otros eventos útiles: `<Button-1>` para clic izquierdo, `<KeyRelease>` tras soltar una tecla. Evita vincular eventos que actualicen el mismo campo sin control, porque pueden provocar cambios repetidos.

## 5. Diálogos, archivos y menús 

`messagebox` muestra información, avisos o errores. `filedialog.askopenfilename()` pide un archivo existente; `asksaveasfilename()` pregunta dónde guardarlo. Si el usuario cancela, la ruta es vacía: compruébala antes de abrir. Para texto usa `encoding="utf-8"` y `try/except OSError` para informar fallos de lectura o escritura. `Menu` organiza comandos en una barra.

**Ejemplo corto:** un editor local de texto. No necesita Internet.

In [3]:
import tkinter as tk
from tkinter import filedialog, messagebox

root = tk.Tk()
root.title("Editor local")
root.geometry("520x340")
texto = tk.Text(root, wrap="word")
texto.pack(fill="both", expand=True)

def abrir():
    ruta = filedialog.askopenfilename(filetypes=[("Texto", "*.txt"), ("Todos", "*.*")])
    if not ruta:
        return
    try:
        with open(ruta, "r", encoding="utf-8") as archivo:
            contenido = archivo.read()
    except (OSError, UnicodeError) as error:
        messagebox.showerror("No se pudo abrir", str(error))
        return
    texto.delete("1.0", "end")
    texto.insert("1.0", contenido)

def guardar():
    ruta = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Texto", "*.txt")])
    if not ruta:
        return
    try:
        with open(ruta, "w", encoding="utf-8") as archivo:
            archivo.write(texto.get("1.0", "end-1c"))
    except OSError as error:
        messagebox.showerror("No se pudo guardar", str(error))
        return
    messagebox.showinfo("Guardado", "Archivo guardado correctamente.")

barra = tk.Menu(root)
archivo_menu = tk.Menu(barra, tearoff=0)
archivo_menu.add_command(label="Abrir...", command=abrir)
archivo_menu.add_command(label="Guardar como...", command=guardar)
barra.add_cascade(label="Archivo", menu=archivo_menu)
root.config(menu=barra)
root.mainloop()

**Comprueba:** guarda un archivo, ciérralo, vuelve a abrirlo y verifica el contenido. Si cancelas el diálogo, el texto debe permanecer intacto.

### Consulta sobre elementos del material original

- `PhotoImage` muestra imágenes PNG o GIF; conserva una referencia Python a la imagen mientras se usa. Es opcional para la actividad principal.
- `ttk.Style` permite cambiar la apariencia; conviene dominar primero el funcionamiento de la interfaz.
- Los menús llaman a las mismas funciones que un botón, por eso se pueden ofrecer dos maneras de ejecutar una acción.

## 6. Proyecto guiado: bloc de notas 

**Meta:** escribir texto, abrirlo y guardarlo; usar menú, barra de desplazamiento y estado. El siguiente ejemplo parte del proyecto del material original. Identifica en el código: ventana, estado (`filepath`, `StringVar`), callbacks y lectura del `Text`. Ejecuta y comprueba en este orden: escribir → guardar como → cambiar texto → guardar → abrir un `.txt`.

In [4]:

import tkinter as tk
from tkinter import filedialog, messagebox, ttk

class BlocNotas:
    def __init__(self, root):
        self.root = root
        self.root.title("Bloc de Notas - Mini Proyecto")
        self.filepath = None

        # Barra de menús
        menubar = tk.Menu(root)
        menu_archivo = tk.Menu(menubar, tearoff=0)
        menu_archivo.add_command(label="Nuevo", command=self.nuevo)
        menu_archivo.add_command(label="Abrir...", command=self.abrir)
        menu_archivo.add_command(label="Guardar", command=self.guardar)
        menu_archivo.add_command(label="Guardar como...", command=self.guardar_como)
        menu_archivo.add_separator()
        menu_archivo.add_command(label="Salir", command=root.quit)
        menubar.add_cascade(label="Archivo", menu=menu_archivo)

        root.config(menu=menubar)

        # Área de texto con scroll
        self.text = tk.Text(root, wrap="word", undo=True)
        scroll = ttk.Scrollbar(root, command=self.text.yview)
        self.text.configure(yscrollcommand=scroll.set)
        self.text.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        # Barra de estado
        self.status = tk.StringVar(value="Listo")
        statusbar = ttk.Label(root, textvariable=self.status, anchor="w")
        statusbar.pack(side="bottom", fill="x")

        # Eventos
        self.text.bind("<<Modified>>", self._on_modified)

    def _on_modified(self, event):
        self.status.set("Modificado...")
        self.text.edit_modified(False)

    def nuevo(self):
        self.text.delete("1.0", "end")
        self.filepath = None
        self.status.set("Nuevo documento")

    def abrir(self):
        ruta = filedialog.askopenfilename(filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        if ruta:
            with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
                contenido = f.read()
            self.text.delete("1.0", "end")
            self.text.insert("1.0", contenido)
            self.filepath = ruta
            self.status.set(f"Abierto: {ruta}")

    def guardar(self):
        if not self.filepath:
            return self.guardar_como()
        with open(self.filepath, "w", encoding="utf-8") as f:
            f.write(self.text.get("1.0", "end-1c"))
        self.status.set(f"Guardado: {self.filepath}")

    def guardar_como(self):
        ruta = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        if ruta:
            self.filepath = ruta
            self.guardar()

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("600x420")
    app = BlocNotas(root)
    root.mainloop()


**Análisis del proyecto:** `self.filepath` recuerda la ruta para que “Guardar” reutilice el archivo; “Guardar como” solicita una ruta nueva. `self.text.get("1.0", "end-1c")` extrae el texto sin salto final. `<<Modified>>` avisa de cambios; `edit_modified(False)` restablece el indicador. Si el usuario cancela el diálogo, la función sigue sin alterar el documento.

**Mejora guiada:** antes de perder cambios, pregunta con `messagebox.askyesnocancel` si quiere guardarlos. El ejemplo base es didáctico y todavía no protege los cambios sin guardar; evita cerrar un texto importante antes de guardar.

**Nota de diseño:** en la aplicación original `pack(side="left")` para el editor y `pack(side="bottom")` para el estado puede dar lugar a una distribución estrecha según el gestor; si sucede, organiza el editor y el scroll dentro de un `ttk.Frame` y empaqueta primero la barra inferior.

## 7. Ejercicios de aplicación 

Trabaja individualmente. Los ejercicios 1–4 sirven para consolidar; 5–7 amplían el proyecto. Entrega el código `.py` o el notebook con capturas o una demostración funcional.

1. **Saludo validado (5 min):** crea `Entry`, botón y `Label`. Si el nombre está vacío, muestra un aviso; si tiene texto, saluda.
2. **Turno de atención (5 min):** crea tres `Radiobutton` y muestra la elección al pulsar “Confirmar”.
3. **Ficha de pedido (8 min):** registra cliente, observaciones en `Text` y entrega a domicilio con `Checkbutton`. Presenta un resumen validado.
4. **Categoría con combo (8 min):** usa `ttk.Combobox` con al menos cuatro categorías. Bloquea el valor libre y exige elegir una opción.
5. **Buscar con Enter (8 min):** reutiliza la lista local de productos; el botón y Enter deben llamar a la misma función. Señala cuántas coincidencias aparecen.
6. **Bloc de notas mejorado (12 min):** añade “Contar palabras” al menú y muestra el número en la barra de estado. Un archivo vacío tiene cero palabras; puedes usar `len(contenido.split())`.
7. **Reto integrador (si queda tiempo o como tarea):** antes de reemplazar un documento al abrir otro o al crear uno nuevo, consulta si debe guardarse. Si cancela, conserva el texto. Considera errores al abrir/guardar y distingue entre “Guardar” y “Guardar como”.

**Criterios de logro:** la interfaz abre y responde; valida datos incompletos; presenta resultados claros; guarda y recupera UTF-8; no modifica el contenido al cancelar. Trabaja en parejas para revisar el reto 7.

## 8. Cierre 

Responde en tu notebook: (1) ¿por qué `Text` necesita índices para `.get()`? (2) ¿qué diferencia existe entre `command=buscar` y `bind("<Return>", buscar)`? (3) ¿qué hace el programa al cancelar “Guardar como”? (4) ¿qué campos validarías en una ficha real de farmacia y por qué?

**Autoevaluación:** puedo crear una ventana, recuperar datos, conectar un evento, seleccionar opciones y guardar un archivo local. **Próxima clase:** diseño de interfaces con `Frame`/`grid`, estilos y validaciones más robustas.